# Loading Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train=pd.read_csv("/kaggle/input/titanic/train.csv")
test=pd.read_csv("/kaggle/input/titanic/test.csv")
sample=pd.read_csv("/kaggle/input/titanic/gender_submission.csv")

In [ ]:
train.shape, test.shape

In [ ]:
train.head()

# Data Prepartion

* **How many passenger are survived?**

In [ ]:
fig = plt.figure(figsize=(3, 3))
labels = 'Survived', 'Not Survived'
sizes = [len(train[train['Survived']==1]), len(train[train['Survived']==0])] 
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
plt.axis('equal')  
plt.show()

* **How many passenger are male and female?**

In [ ]:
fig = plt.figure(figsize=(3, 3))
labels = 'Male', 'Female'
sizes = [len(train[train['Sex']=='male']), len(train[train['Sex']=='female'])] 
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
plt.axis('equal')  
plt.show()

In [ ]:
children = train[train['Age'].notna() & (train['Age'] < 18)]
adults = train[train['Age'].notna() & (train['Age'] >= 18) & (train['Age'] < 60)]
old = train[train['Age'].notna() & (train['Age'] >= 60)]

fig = px.pie(
    values=[len(children), len(adults), len(old)],
    names=['Child', 'Adult', 'Old'],
    hover_name=['Child', 'Adult', 'Old'],
    title='Category classification',
    opacity=0.9,
    template='simple_white',
)
fig.show()

In [ ]:
total_passengers = len(train)
total_survived = len(train[train['Survived'] == 1])

child_survival = children[children['Survived'] == 1]
adult_survival = adults[adults['Survived'] == 1]
old_survival = old[old['Survived'] == 1]

In [ ]:
fig = make_subplots(rows=1, cols=4,
                    specs=[[{'type':'domain'}, {'type':'domain'}, {'type':'domain'}, {'type':'domain'}]],
                    subplot_titles=["Children", "Adults", "Old", "Overall"])

fig.add_trace(go.Pie(
    labels=["Total Child", "Survived"],
    values=[len(children), len(child_survival)],
    name="Children"
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=["Total Adult", "Survived"],
    values=[len(adults), len(adult_survival)],
    name="Adults"
), row=1, col=2)

fig.add_trace(go.Pie(
    labels=["Total Old", "Survived"],
    values=[len(old), len(old_survival)],
    name="Old"
), row=1, col=3)

fig.add_trace(go.Pie(
    labels=["Total", "Survived"],
    values=[total_passengers, total_survived],
    name="Overall"
), row=1, col=4)

fig.update_layout(
    title="Total vs Survival Rate by Category (Pie Charts)",
    height=700,
    template="simple_white"
)

fig.show()

In [ ]:
upper_class = train[train['Pclass']==1]
middle_class = train[train['Pclass']==2]
lower_class = train[train['Pclass']==3]

upper_class_survived = train[(train['Pclass']==1) & (train['Survived']==1)]
middle_class_survived = train[(train['Pclass']==2) & (train['Survived']==1)]
lower_class_survived = train[(train['Pclass']==3) & (train['Survived']==1)]

fig = make_subplots(rows=1, cols=2,
                    specs=[[{'type':'domain'}, {'type':'domain'}]],
                    subplot_titles=['Passenger Class', 'Passenger Survived'])

fig.add_trace(go.Pie(
    labels=['Upper Class', 'Middle Class', 'Lower Class'],
    values=[len(upper_class), len(middle_class), len(lower_class)],
    name='Passenger'
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=['Upper Class Survived', 'Middle Class Survived', 'Lower Class Survived'],
    values=[len(upper_class_survived), len(middle_class_survived), len(lower_class_survived)],
    name='Passenger Survived'
), row=1, col=2)

fig.update_layout(
    title='Total Passenger vs Passenger Survived by Class',
    template='simple_white'
)

fig.show()


In [ ]:
not_alone = train[(train['SibSp']>0) | (train['Parch']>0)]
alone = train[(train['SibSp']==0) & (train['Parch']==0)]

fig = make_subplots(rows=1, cols=2,
                    specs=[[{'type':'domain'}, {'type':'domain'}]],
                    subplot_titles=['With family', 'alone'])

fig.add_trace(go.Pie(
    labels=['Total Passengers', 'With Family'],
    values=[len(train), len(not_alone)],
    name='Passenger'
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=['Alone', 'With Family'],
    values=[len(alone), len(not_alone)],
    name='Passenger'
), row=1, col=2)

fig.update_layout(
    title='Total Passengers vs With Family vs Alone',
    template="simple_white"
)

fig.show()

* **How many people boarded from which place?** 

In [ ]:
S_df = train[train['Embarked']=='S']
C_df = train[train['Embarked']=='C']
Q_df = train[train['Embarked']=='Q']

fig = px.pie(
    values=[len(S_df), len(C_df), len(Q_df)],
    title='Boarded from which place',
    names=['Southampton (England)', 'Cherbourg (France)', 'Queenstown (Ireland)'],
    hover_name=['Southampton (England)', 'Cherbourg (France)', 'Queenstown (Ireland)'],
    opacity=.9,
    template='simple_white'
)
fig.show()

In [ ]:
train.isna().sum()

In [ ]:
train.info()

# Model Evaluation

In [ ]:
def Encoder(df):
    le = LabelEncoder()

    df.dropna(subset=['Embarked'], inplace=True)
    mean_fare=df['Fare'].mean()
    df['Fare'].fillna(mean_fare, inplace=True)
    mean_age = df['Age'].mean()
    df['Age'].fillna(mean_age, inplace=True)

    df['Sex'] = le.fit_transform(df['Sex'])
    df['Embarked'] = le.fit_transform(df['Embarked'])

    df.drop('Cabin', axis=1, inplace=True)
    df.drop('Ticket', axis=1, inplace=True)
    df.drop("Name", axis=1, inplace=True)

In [ ]:
Encoder(train)
Encoder(test)

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
x_train=train.drop("Survived", axis=1)
y_train=train["Survived"]
x_test=test

In [ ]:
model=LogisticRegression()

In [ ]:
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': y_pred
})
output.to_csv('submission.csv', index=False)

In [ ]:
output.head()